# Desafío por equipos · Orquestación y gobierno de datos 

**Caso:** NexoServicios · **Siete equipos, siete desafíos distintos.** Cada equipo resuelve el suyo.

Todos los desafíos **integran los temas de la clase**: orquestación, buenas prácticas y errores comunes, gobierno de datos y calidad. Cada equipo entrega:

1. **Una DAG comentada**
2. **Una matriz RACI breve**
3. **Controles bloqueantes y no bloqueantes**

Y, dentro de eso, la **decisión de usar o no un orquestador** con la alternativa descartada, y el **plan ante una falla**.

**Una regla:** cada decisión incluye **una alternativa que se descartó**. Si preguntan «¿está bien?», la respuesta del facilitador va a ser: *¿qué restricción satisface y cómo comprobarías que la satisface?*

Los datos son ficticios. No hay credenciales ni infraestructura real.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook; si ya está todo, no hace nada.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "jinja2": "jinja2"}
INSTALAR_XGBOOST = False          # opcional: ponelo en True para sumar XGBoost a la comparación de modelos
if INSTALAR_XGBOOST: REQUISITOS["xgboost"] = "xgboost"

print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")


## Preparación

In [ ]:
# Este archivo es autocontenido: instala solo lo que le falte (se corre una sola vez)
import importlib.util
import subprocess
import sys

LIBRERIAS = ("numpy", "pandas")
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    for extra in ([], ["--user"], ["--break-system-packages"]):          # si un intento falla, prueba el siguiente
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *faltan], capture_output=True, text=True)
        if r.returncode == 0:
            break
    else:
        print("⚠️ No se pudo instalar solo. Corré en una celda:  %pip install " + " ".join(faltan))
        print(r.stderr[-300:])
    importlib.invalidate_caches()
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
print("✅ Librerías listas" if not faltan else f"⚠️ Todavía faltan: {faltan}")

In [ ]:
# ═══════════════════════════ KIT DE PIPELINES · orquestación, calidad y gobierno ═══════════════════════════
# No hace falta instalar Airflow: este mini orquestador simula sus conceptos principales
# (DAG, tareas, dependencias, programación, reintentos, sensores, backfill) para poder razonar y probar el diseño.
import hashlib
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
from IPython.display import display

# ───────────────────────── 1 · EL ORQUESTADOR ─────────────────────────
class ErrorTransitorio(Exception):
    """Falla que puede resolverse reintentando: un timeout, un límite de tasa."""
class ControlFallido(Exception):
    """Un control bloqueante no pasó: reintentar no lo arregla."""
class SensorVencido(Exception):
    """Lo que se esperaba (un archivo, un dato) no llegó dentro del plazo."""

@dataclass
class Tarea:
    nombre: str
    funcion: object
    depende_de: tuple = ()
    reintentos: int = 2
    espera_s: float = 30.0
    siempre: bool = False          # como trigger_rule="all_done": corre aunque falle algo antes
    descripcion: str = ""

def nuevo_contexto(fecha, almacen=None, **extra):
    """El contexto es lo que viaja por la DAG: la fecha de la corrida, el almacén de destino, los datos y las alertas."""
    return {"fecha": pd.Timestamp(fecha), "almacen": {} if almacen is None else almacen, "datos": {}, "alertas": [], **extra}

class Corrida:
    def __init__(self, dag, fecha):
        self.dag, self.fecha, self.filas = dag, pd.Timestamp(fecha), []
    @property
    def log(self):
        return pd.DataFrame(self.filas, columns=["tarea", "estado", "intentos", "espera simulada (s)", "filas", "detalle"])
    @property
    def exitosa(self):
        return all(f["estado"] in ("exitosa", "alerta") for f in self.filas)
    def estado(self, tarea):
        return next(f["estado"] for f in self.filas if f["tarea"] == tarea)

class Dag:
    """Una DAG: tareas con dependencias, sin ciclos, con una programación y una política de reintentos."""
    def __init__(self, nombre, programacion="0 6 * * *", reintentos=2, espera_s=30.0, responsable="(sin responsable)"):
        self.nombre, self.programacion, self.reintentos, self.espera_s, self.responsable = nombre, programacion, reintentos, espera_s, responsable
        self.tareas = {}

    def tarea(self, nombre=None, depende_de=(), reintentos=None, espera_s=None, siempre=False, descripcion=""):
        """Decorador que registra una función como tarea de la DAG."""
        def registrar(funcion):
            n = nombre or funcion.__name__
            self.tareas[n] = Tarea(n, funcion, tuple(depende_de), self.reintentos if reintentos is None else reintentos,
                                   self.espera_s if espera_s is None else espera_s, siempre, descripcion or (funcion.__doc__ or "").strip())
            return funcion
        return registrar

    def orden(self):
        """Orden de ejecución (orden topológico). Falla si hay una dependencia inexistente o un ciclo."""
        for t in self.tareas.values():
            for d in t.depende_de:
                if d not in self.tareas:
                    raise ValueError(f"La tarea «{t.nombre}» depende de «{d}», que no existe")
        pendientes, listas = dict(self.tareas), []
        while pendientes:
            libres = [n for n, t in pendientes.items() if all(d in listas for d in t.depende_de)]
            if not libres:
                raise ValueError(f"Hay un ciclo entre las tareas: {sorted(pendientes)}")
            listas.append(libres[0]); pendientes.pop(libres[0])
        return listas

    def dibujar(self):
        """Muestra la DAG como niveles: lo que puede correr en paralelo queda en el mismo nivel."""
        nivel = {}
        for n in self.orden():
            nivel[n] = 1 + max([nivel[d] for d in self.tareas[n].depende_de], default=0)
        print(f"DAG «{self.nombre}» · programación: {self.programacion} · responsable: {self.responsable}")
        for k in sorted(set(nivel.values())):
            print(f"  nivel {k}: " + "   ".join(f"[{n}]" + (f" ← {', '.join(self.tareas[n].depende_de)}" if self.tareas[n].depende_de else "") for n in nivel if nivel[n] == k))

    def correr(self, fecha, ctx=None, fallas=None, mostrar=True):
        """Corre la DAG para una fecha. `fallas` inyecta fallas para probar el diseño:
        {"tarea": 2} falla las dos primeras veces; {"tarea": "siempre"} falla siempre."""
        ctx = ctx if ctx is not None else nuevo_contexto(fecha)
        ctx["fecha"] = pd.Timestamp(fecha)
        pendientes = dict(fallas or {})
        corrida = Corrida(self, fecha)
        estados = {}
        for n in self.orden():
            t = self.tareas[n]
            rotas = [d for d in t.depende_de if estados[d] in ("fallida", "omitida")]
            if rotas and not t.siempre:
                estados[n] = "omitida"
                corrida.filas.append({"tarea": n, "estado": "omitida", "intentos": 0, "espera simulada (s)": 0, "filas": None, "detalle": f"no corrió: falló {', '.join(rotas)}"})
                continue
            alertas_antes, espera, estado, detalle, filas, intentos = len(ctx["alertas"]), 0.0, "exitosa", "", None, 0
            for intento in range(1, t.reintentos + 2):
                intentos = intento
                try:
                    if pendientes.get(n) == "siempre" or (isinstance(pendientes.get(n), int) and pendientes[n] > 0):
                        if isinstance(pendientes[n], int):
                            pendientes[n] -= 1
                        raise ErrorTransitorio("falla inyectada (timeout)")
                    resultado = t.funcion(ctx)
                    filas = resultado if isinstance(resultado, (int, np.integer)) else None
                    estado = "alerta" if len(ctx["alertas"]) > alertas_antes else "exitosa"
                    detalle = "; ".join(ctx["alertas"][alertas_antes:])
                    if intento > 1:
                        detalle = f"recuperó en el intento {intento}" + (f"; {detalle}" if detalle else "")
                    break
                except (ControlFallido, SensorVencido) as e:
                    estado, detalle = "fallida", str(e)
                    break
                except Exception as e:
                    detalle = f"{type(e).__name__}: {e}"
                    if intento <= t.reintentos:
                        espera += t.espera_s * 2 ** (intento - 1)         # espera creciente entre reintentos
                    else:
                        estado = "fallida"
            estados[n] = estado
            corrida.filas.append({"tarea": n, "estado": estado, "intentos": intentos, "espera simulada (s)": espera, "filas": filas, "detalle": detalle})
        if mostrar:
            display(corrida.log)
            print(f"Corrida del {pd.Timestamp(fecha).date()}: {'EXITOSA' if corrida.exitosa else 'CON FALLAS'}")
        return corrida

    def backfill(self, desde, hasta, almacen=None, fallas=None):
        """Reprocesa un rango de fechas, una corrida por día. Sirve solo si las tareas son idempotentes."""
        almacen = {} if almacen is None else almacen
        filas = []
        for f in pd.date_range(desde, hasta, freq="D"):
            c = self.correr(f, nuevo_contexto(f, almacen), fallas, mostrar=False)
            filas.append({"fecha": f.date(), "resultado": "exitosa" if c.exitosa else "con fallas", "tareas fallidas": ", ".join(r["tarea"] for r in c.filas if r["estado"] == "fallida")})
        return pd.DataFrame(filas), almacen

# ───────────────────────── 2 · CALIDAD: CONTROLES ─────────────────────────
@dataclass
class Control:
    nombre: str
    dimension: str                # completitud, unicidad, validez, consistencia, frescura, volumen, integridad, seguridad
    regla: object                 # función: tabla -> (pasó, detalle)
    severidad: str = "bloqueante" # "bloqueante" detiene lo que sigue; "no bloqueante" avisa y sigue
    accion: str = ""              # qué se hace cuando no pasa
    responsable: str = ""         # quién decide y actúa

def c_esquema(columnas, **k):
    return Control("esquema esperado", "consistencia", lambda d: (set(columnas) <= set(d.columns), f"faltan {sorted(set(columnas) - set(d.columns))}; sobran {sorted(set(d.columns) - set(columnas))}" if set(columnas) != set(d.columns) else "columnas completas"), **k)
def c_sin_nulos(columnas, max_pct=0.0, **k):
    def regla(d):
        pct = 100 * d[columnas].isna().any(axis=1).mean() if len(d) else 0
        return pct <= max_pct, f"{pct:.1f} % de filas con nulos en {columnas} (máximo {max_pct} %)"
    return Control(f"sin nulos en {', '.join(columnas)}", "completitud", regla, **k)
def c_clave_unica(columna, **k):
    def regla(d):
        rep = int(d[columna].duplicated().sum())
        return rep == 0, f"{rep} valores repetidos en «{columna}»"
    return Control(f"clave única «{columna}»", "unicidad", regla, **k)
def c_rango(columna, minimo, maximo, max_pct=0.0, **k):
    def regla(d):
        fuera = 100 * (~d[columna].between(minimo, maximo)).mean() if len(d) else 0
        return fuera <= max_pct, f"{fuera:.1f} % de «{columna}» fuera de [{minimo}, {maximo}]"
    return Control(f"rango de «{columna}»", "validez", regla, **k)
def c_valores(columna, permitidos, **k):
    def regla(d):
        malos = d.loc[~d[columna].isin(permitidos), columna].unique().tolist()
        return not malos, f"valores no permitidos: {malos}" if malos else "valores válidos"
    return Control(f"valores válidos de «{columna}»", "validez", regla, **k)
def c_frescura(columna_fecha, fecha_esperada, max_dias=0, **k):
    def regla(d):
        ultima = pd.to_datetime(d[columna_fecha]).max() if len(d) else pd.NaT
        ok = pd.notna(ultima) and (pd.Timestamp(fecha_esperada) - ultima).days <= max_dias
        return ok, f"último dato: {ultima.date() if pd.notna(ultima) else 'sin datos'} · esperado: {pd.Timestamp(fecha_esperada).date()}"
    return Control(f"frescura de «{columna_fecha}»", "frescura", regla, **k)
def c_volumen(esperado, tolerancia=0.3, **k):
    def regla(d):
        var = abs(len(d) - esperado) / esperado
        return var <= tolerancia, f"{len(d)} filas contra {esperado} esperadas ({100 * var:.0f} % de diferencia; tolerancia {100 * tolerancia:.0f} %)"
    return Control("volumen razonable", "volumen", regla, **k)
def c_integridad(columna, referencia, **k):
    def regla(d):
        huerfanos = int((~d[columna].isin(referencia)).sum())
        return huerfanos == 0, f"{huerfanos} filas con «{columna}» que no existe en la tabla de referencia"
    return Control(f"integridad de «{columna}»", "integridad", regla, **k)
def c_sin_pii(columnas_sensibles, **k):
    def regla(d):
        expuestas = [c for c in columnas_sensibles if c in d.columns]
        return not expuestas, f"columnas sensibles expuestas: {expuestas}" if expuestas else "sin columnas sensibles"
    return Control("sin datos sensibles expuestos", "seguridad", regla, **k)

def evaluar_controles(tabla, controles, ctx=None, cortar=True):
    """Corre los controles sobre una tabla. Si falla uno BLOQUEANTE y `cortar=True`, levanta ControlFallido
    (la DAG detiene lo que depende de esta tarea). Los NO BLOQUEANTES dejan una alerta y la DAG sigue."""
    filas = []
    for c in controles:
        try:
            ok, detalle = c.regla(tabla)
        except KeyError as e:                                    # si la columna ya no existe, el control falla con un mensaje claro
            ok, detalle = False, f"no se pudo evaluar: falta la columna {e}"
        filas.append({"control": c.nombre, "dimensión": c.dimension, "severidad": c.severidad, "pasó": bool(ok), "detalle": detalle, "acción si falla": c.accion, "responsable": c.responsable})
    informe = pd.DataFrame(filas)
    if ctx is not None:
        ctx.setdefault("controles", []).append(informe)
        for f in filas:
            if not f["pasó"] and f["severidad"] != "bloqueante":
                ctx["alertas"].append(f"{f['control']}: {f['detalle']}")
    bloqueantes = informe[(~informe["pasó"]) & (informe["severidad"] == "bloqueante")]
    if cortar and len(bloqueantes):
        raise ControlFallido("; ".join(f"{r['control']} → {r['detalle']}" for _, r in bloqueantes.iterrows()))
    return informe

# ───────────────────────── 3 · GOBIERNO ─────────────────────────
def enmascarar(tabla, columnas, sal="nexo"):
    """Reemplaza valores sensibles por un código irreversible. El mismo valor da siempre el mismo código, así se puede unir sin exponerlo."""
    t = tabla.copy()
    for c in columnas:
        t[c] = t[c].astype(str).map(lambda v: hashlib.sha256((sal + v).encode()).hexdigest()[:10])
    return t

def contrato_datos(nombre, owner, steward, columnas, frescura_horas, consumidores, controles=()):
    """Un contrato de datos: qué es, quién responde, qué contiene, qué tan fresco debe estar y quién lo consume.
    `columnas` es {nombre: (tipo, clasificación)}, con clasificación pública, interna o sensible."""
    return {"nombre": nombre, "owner": owner, "steward": steward, "columnas": columnas, "frescura_horas": frescura_horas,
            "consumidores": list(consumidores), "controles": [c.nombre for c in controles]}

def ver_contrato(c):
    print(f"CONTRATO «{c['nombre']}» · owner: {c['owner']} · steward: {c['steward']} · frescura máxima: {c['frescura_horas']} h · consumidores: {', '.join(c['consumidores'])}")
    t = pd.DataFrame([{"columna": k, "tipo": v[0], "clasificación": v[1]} for k, v in c["columnas"].items()])
    display(t)
    if c["controles"]:
        print("Controles asociados:", "; ".join(c["controles"]))

def raci(actividades, roles=("Arquitectura", "Operación", "Gobierno", "Revisión")):
    """Matriz RACI: R responsable, A quien rinde cuentas (uno solo por fila), C consultado, I informado.
    `actividades` es {actividad: {rol: "R"/"A"/"A/R"/"C"/"I"}}. Valida que haya exactamente una A por actividad."""
    filas = []
    for act, asignacion in actividades.items():
        cantidad_a = sum("A" in v for v in asignacion.values())
        if cantidad_a != 1:
            raise ValueError(f"«{act}» tiene {cantidad_a} roles con A: debe haber exactamente uno")
        filas.append({"actividad": act, **{r: asignacion.get(r, "") for r in roles}})
    return pd.DataFrame(filas)

# ───────────────────────── 4 · EL CASO NEXOSERVICIOS (datos ficticios) ─────────────────────────
def fuentes_nexo(fecha, n_ordenes=480):
    """Genera las fuentes de NexoServicios para una fecha: operaciones (transaccional), CRM, archivo de un proveedor y API de entregas."""
    fecha = pd.Timestamp(fecha)
    azar = np.random.default_rng(int(fecha.strftime("%Y%m%d")))
    base = np.random.default_rng(2026)                                   # el CRM es el mismo todos los días
    clientes = pd.DataFrame({"id_cliente": [f"C{i:04d}" for i in range(1, 301)],
                             "nombre": [f"Cliente {i:04d}" for i in range(1, 301)],
                             "email": [f"cliente{i:04d}@ejemplo.com" for i in range(1, 301)],
                             "telefono": [f"11-5555-{i:04d}" for i in range(1, 301)],
                             "dni": base.integers(10_000_000, 49_999_999, 300).astype(str),
                             "segmento": base.choice(["A", "B", "C"], 300, p=[.2, .5, .3])})
    n = int(azar.integers(int(n_ordenes * 0.9), int(n_ordenes * 1.1)))
    ordenes = pd.DataFrame({"id_orden": [f"O-{fecha:%Y%m%d}-{i:04d}" for i in range(1, n + 1)], "fecha": fecha.date().isoformat(),
                            "id_cliente": azar.choice(clientes["id_cliente"], n), "canal": azar.choice(["web", "tienda", "telefono"], n, p=[.5, .3, .2]),
                            "monto": np.clip(azar.lognormal(7.0, 0.9, n), 50, 48_000).round(2),
                            "estado": azar.choice(["abierta", "asignada", "cerrada"], n, p=[.15, .25, .6])})
    proveedor = ordenes.sample(frac=0.7, random_state=int(fecha.strftime("%Y%m%d")))[["id_orden"]].assign(costo_proveedor=lambda d: (azar.uniform(0.4, 0.7, len(d)) * ordenes.set_index("id_orden").loc[d["id_orden"], "monto"].to_numpy()).round(2))
    entregas = ordenes[["id_orden"]].assign(estado_entrega=azar.choice(["pendiente", "en camino", "entregada"], n, p=[.2, .3, .5]))
    return {"operaciones": ordenes, "crm": clientes, "proveedor": proveedor.reset_index(drop=True), "entregas": entregas}

# ───────────────────────── 5 · REVISORES: ¿el diseño está completo? ─────────────────────────
def revisar_controles(controles):
    """Marca los controles que registran un fallo pero no dicen qué hacer ni quién decide."""
    filas = []
    for c in controles:
        faltas = []
        if not c.accion.strip():
            faltas.append("no dice qué hacer si falla")
        if not c.responsable.strip():
            faltas.append("no dice quién decide")
        if c.severidad not in ("bloqueante", "no bloqueante"):
            faltas.append("severidad inválida")
        filas.append({"control": c.nombre, "severidad": c.severidad, "estado": "✔ completo" if not faltas else "✘ " + "; ".join(faltas)})
    tabla = pd.DataFrame(filas)
    display(tabla)
    return tabla

def revisar_dag(dag, controles=()):
    """Revisa un diseño de DAG contra una lista corta de criterios de calidad."""
    cumple = {
        "Tiene responsable": dag.responsable != "(sin responsable)",
        "Tiene programación": bool(dag.programacion),
        "Todas las tareas están descritas": all(t.descripcion for t in dag.tareas.values()),
        "Hay al menos un control de calidad entre las tareas": any("control" in n for n in dag.tareas),
        "Hay una tarea que avisa aunque algo falle": any(t.siempre for t in dag.tareas.values()),
        "Los reintentos están acotados (5 o menos)": all(t.reintentos <= 5 for t in dag.tareas.values()),
        "Hay controles bloqueantes y no bloqueantes": {"bloqueante", "no bloqueante"} <= {c.severidad for c in controles},
        "Cada control dice qué hacer y quién decide": all(c.accion.strip() and c.responsable.strip() for c in controles) and len(controles) > 0,
    }
    for criterio, ok in cumple.items():
        print("✔" if ok else "✘", criterio)
    return all(cumple.values())


COLUMNAS_OPERACIONES = ["id_orden", "fecha", "id_cliente", "canal", "monto", "estado"]
SENSIBLES = ["nombre", "email", "telefono", "dni"]

## Tu caso

Cambiá `MI_EQUIPO` por el número de tu equipo (1 a 7) y corré la celda. Te muestra la situación, lo que el caso **declara** y lo que **no declara** (eso lo tienen que preguntar o asumir de forma explícita), y el error a evitar.

In [ ]:
ENTREGABLE = ['Una DAG comentada: tareas, dependencias, programación y política de reintentos', 'Una matriz RACI breve, con una sola A por actividad', 'Controles bloqueantes y no bloqueantes (al menos tres y dos), cada uno con acción y responsable', 'La decisión de usar o no un orquestador, con la alternativa que se descartó y por qué', 'El plan ante una falla: cómo se detecta, se contiene y se recupera, y quién decide']

In [ ]:
CASOS = {
    1: dict(titulo='Indicadores diarios antes de las 8:00', situacion='Dirección necesita los indicadores de ayer antes de las 8:00. Las fuentes son las operaciones, el CRM y un archivo externo, y el equipo tiene tres personas.', declarado='Ventana de 6:00 a 8:00. Tres fuentes con horarios distintos.', no_declarado='A qué hora llega cada fuente y qué se publica si una llega a las 7:50.', foco='Dependencias, programación y qué se publica si una fuente llega tarde.', cuidado='Elegir la herramienta antes de fijar la ventana.', debrief='¿Qué decisión cambiaría si la ventana se redujera diez veces?'),
    2: dict(titulo='Alertas durante el día para Operaciones', situacion='Operaciones quiere enterarse en menos de 15 minutos si se acumulan órdenes sin asignar. Hoy mira un reporte una vez por día.', declarado='Latencia de 15 minutos. Las órdenes llegan al sistema transaccional todo el día.', no_declarado='Cuántas órdenes sin asignar son una «acumulación» y quién recibe la alerta.', foco='La frecuencia sale del requisito: lotes frecuentes, sensor o eventos. Costo de cada opción.', cuidado='Pedir tiempo real sin fijar el requisito de latencia.', debrief='¿Qué cambia si el requisito pasa de 15 minutos a 15 segundos?'),
    3: dict(titulo='El archivo del proveedor llega tarde o repetido', situacion='Un proveedor sube un archivo entre las 5:00 y las 9:00. A veces llega dos veces, la segunda con correcciones, y a veces no llega.', declarado='Los indicadores se publican a las 8:00.', no_declarado='Cuánto retraso tolera el negocio y si la segunda versión reemplaza a la primera.', foco='Sensor con plazo, versión del archivo, idempotencia y qué hacer si no llega.', cuidado='Reprocesar el archivo corregido y duplicar lo que ya se cargó.', debrief='¿Qué evidencia necesita una persona ajena para saber qué versión se usó?'),
    4: dict(titulo='La API con límite y caídas', situacion='La API de entregas permite 1.000 llamadas por hora y cae unos minutos por semana. Hay que traer un día completo y, a veces, reprocesar una semana.', declarado='Límite de 1.000 llamadas por hora.', no_declarado='Cuántas llamadas necesita un día y qué pasa al reprocesar varios días.', foco='Reintentos con espera creciente, avance por tandas con punto de control y reproceso sin saturar la fuente.', cuidado='Reintentar sin límite y agotar la cuota.', debrief='¿Qué componente concentra el mayor radio de impacto si la API cae?'),
    5: dict(titulo='Datos sensibles en el camino', situacion='El CRM trae nombre, mail, teléfono y DNI. Los indicadores de Dirección no los necesitan, pero Atención al Cliente sí necesita ver el teléfono de un cliente puntual.', declarado='Los datos sensibles no pueden llegar a las tablas de indicadores.', no_declarado='Quién autoriza el acceso de Atención al Cliente y cuánto tiempo se conservan los datos.', foco='Clasificación, enmascarado, acceso por rol, control bloqueante de seguridad y registro de quién accede.', cuidado='Esconder una decisión humana, quién autoriza, detrás de una automatización.', debrief='¿Qué evidencia necesita una persona ajena para saber quién vio qué?'),
    6: dict(titulo='El esquema cambió y nadie avisó', situacion='Un proveedor renombró una columna y cambió el tipo de otra. La DAG corrió sin error y los indicadores salieron mal durante tres días.', declarado='No se puede obligar al proveedor a avisar antes.', no_declarado='Quién es el dueño de la relación con el proveedor y qué se hace con los tres días ya publicados.', foco='Contrato de datos, control de esquema bloqueante, versión, aviso al productor y recuperación.', cuidado='Definir la calidad solo como «no hay nulos».', debrief='¿Qué control podría generar falsos positivos y cómo se administraría?'),
    7: dict(titulo='¿Hace falta un orquestador?', situacion='Un reporte semanal corre con un único script, tiene un solo dueño y tarda dos minutos. Alguien propone migrarlo a Airflow «para profesionalizarlo».', declarado='El equipo es pequeño y no tiene a nadie que opere la plataforma.', no_declarado='Qué falla hoy, cuánto cuesta una falla y si hay más procesos por venir.', foco='Decidir con criterio. Si es no, qué se agrega: controles, alertas y un procedimiento. Si es sí, qué cambia.', cuidado='Elegir una herramienta antes de fijar el requisito.', debrief='¿Qué parte del diseño depende de una infraestructura que todavía no está confirmada?'),
}

MI_EQUIPO = 1          # <- el número de tu equipo, del 1 al 7

c = CASOS[MI_EQUIPO]
print(f"DESAFÍO {MI_EQUIPO} · {c['titulo']}\n")
print("Situación:    ", c["situacion"])
print("El caso declara:   ", c["declarado"])
print("El caso NO declara:", c["no_declarado"], " ← declaren su supuesto")
print("Foco:         ", c["foco"])
print("Cuidado con:  ", c["cuidado"])
print("\nQué entregar:")
for e in ENTREGABLE: print("  •", e)

## Las herramientas del kit

Son las de la demo. **Calculan y simulan, pero no deciden**: el diseño lo define su equipo.

| Para… | Usá… |
| --- | --- |
| Armar y correr la DAG | `Dag(...)`, `@dag.tarea(...)`, `dag.dibujar()`, `dag.correr(fecha, ctx, fallas=...)`, `dag.backfill(...)` |
| Los datos de NexoServicios | `fuentes_nexo(fecha)`, `nuevo_contexto(fecha)` |
| Controles de calidad | `c_esquema`, `c_sin_nulos`, `c_clave_unica`, `c_rango`, `c_valores`, `c_frescura`, `c_volumen`, `c_integridad`, `c_sin_pii`, `evaluar_controles` |
| Gobierno | `contrato_datos`, `ver_contrato`, `enmascarar`, `raci` |
| Revisar el diseño | `revisar_controles(...)`, `revisar_dag(dag, controles)` |

## Tu turno

### Paso 1 · El problema y la decisión

Escriban, en esta celda de texto, **qué problema y qué decisión resuelve** su diseño, **qué entra y qué queda fuera de alcance**, y **los supuestos** que el caso no declara.

- **Problema y decisión:**
- **Entra:**
- **Fuera de alcance:**
- **Supuestos que declaramos:**

### Paso 2 · ¿Usamos un orquestador?

Decidan **sí o no**, y **qué alternativa descartaron y por qué**. Usen el criterio de la clase:

| Pregunta | Sí | No |
| --- | --- | --- |
| ¿Hay más de una tarea con dependencias? | Seguí | Un `cron` o un script con alerta alcanza |
| ¿Necesita respuesta continua en segundos o minutos? | Procesamiento por eventos o streaming; el orquestador coordina solo lo que va por lotes | Seguí |
| ¿Hay reintentos, reprocesos, alertas o varios responsables? | Un orquestador se justifica | Un `cron` con controles alcanza |

- **Decisión:**
- **Alternativa que descartamos y por qué:**
- **Costo que aceptamos:**

### Paso 3 · La DAG comentada

Armen la DAG: tareas, **dependencias**, **programación**, **reintentos** y un **responsable**. Cada tarea lleva una **descripción** de lo que hace. Pueden ser pocas tareas si el caso lo justifica.

In [ ]:
# 👉 TU TURNO · Paso 3: la DAG comentada
# dag = Dag("nombre_de_la_dag", programacion="0 6 * * *", reintentos=2, responsable="¿quién responde cuando falla?")
#
# @dag.tarea(descripcion="qué hace esta tarea")
# def primera_tarea(ctx):
#     ...
#
# @dag.tarea(depende_de=["primera_tarea"], descripcion="qué hace esta otra")
# def segunda_tarea(ctx):
#     ...
#
# dag.dibujar()

### Paso 4 · Los controles

Definan **al menos tres bloqueantes y dos no bloqueantes**. Cada uno dice **qué hacer si falla** y **quién decide**. Un control que solo registra el fallo no alcanza.

In [ ]:
# 👉 TU TURNO · Paso 4: los controles
# controles = [
#     c_clave_unica("id_orden", severidad="bloqueante", accion="...", responsable="..."),
#     c_volumen(480, 0.3, severidad="no bloqueante", accion="...", responsable="..."),
# ]
# _ = revisar_controles(controles)

### Paso 5 · La matriz RACI

Para cada actividad: **R** (hace), **A** (rinde cuentas: **uno solo**), **C** (consultado), **I** (informado). Los roles son **arquitectura, operación, gobierno y revisión**. Una de las filas tiene que responder **quién decide cuando un control no pasa**.

In [ ]:
# 👉 TU TURNO · Paso 5: la matriz RACI
# matriz = raci({
#     "Una actividad": {"Arquitectura": "R", "Operación": "C", "Gobierno": "A", "Revisión": "I"},
# })
# matriz

### Paso 6 · El plan ante una falla

Completen: **cómo se detecta, se contiene y se recupera**, y **quién decide**. Acuérdense: *monitorear no es recuperar*.

| Etapa | En nuestro caso |
| --- | --- |
| **Detectar** | |
| **Contener** | |
| **Recuperar** | |
| **Decide** | |

### Paso 7 · Probar antes de producción

Simulen **una falla** con el parámetro `fallas` y comprueben que el diseño se comporta como dijeron: ¿qué queda sin correr? ¿qué se publica? ¿quién se entera?

In [ ]:
# 👉 TU TURNO · Paso 7: simular una falla
# corrida = dag.correr("2026-10-15", nuevo_contexto("2026-10-15"), fallas={"nombre_de_una_tarea": "siempre"})

### Paso 8 · La revisión final

Antes de entregar, corran la revisión automática. Cada `✘` es una pregunta que el equipo revisor les va a hacer.

In [ ]:
# 👉 TU TURNO · Paso 8: revisar el diseño
# _ = revisar_dag(dag, controles)

    ## Ficha de entrega

    **Equipo: ________ · Desafío: ________**

    Respondan, en una frase cada una:

    1. **¿Qué problema y qué decisión resuelve?**
    2. **¿Qué entra y qué queda fuera de alcance?**
    3. **¿Qué alternativa se descartó y por qué?**
    4. **¿Cómo se prueba antes de pasar a producción?**
    5. **¿Cómo se detecta, se contiene y se recupera una falla?**
    6. **¿Quién decide cuando un control no pasa?**

    ### Cómo se evalúa

| Nivel | Qué se ve |
| --- | --- |
| **Insuficiente** | Dibuja herramientas sin responsables ni flujos; no explica cómo se valida ni se recupera |
| **Inicial** | Técnicamente posible, pero no dice cómo se prueba, cómo se recupera ni quién decide |
| **Sólido** | Hay trazabilidad entre el problema y el diseño, trade-offs explícitos y controles con acción y responsable |
| **Defendible** | Todo lo anterior, más una alternativa descartada con su costo, una falla simulada y una defensa clara |

Se evalúa **trazabilidad** entre problema y diseño, **manejo de trade-offs**, **verificabilidad**, **operación segura** y **claridad de la defensa**.